#### Import packages

In [145]:
import os
import numpy as np
import time
import csv 
import subprocess

import motor_utils
import flipper_utils

#### Motor settings

In [136]:
def set_motor_angle(ang_1:float, ang_2:float):
    #connect to motor
    motor_utils.init('COM6')
    motor_utils.setup()

    #set angles
    motor_utils.move(position=ang_1, MOTOR_ID=1, speed=20)
    time.sleep(1)
    motor_utils.move(position=ang_2, MOTOR_ID=2, speed=20)
    time.sleep(1)

    #close connection to motor
    motor_utils.deinit()

set_motor_angle(90,90)

m1ang_init = 8.5  # 0 angle of motor 1
m2ang_init = 0.98  # 0 angle of motor 2

Opening COM6...
Port open: True
Initialization commands sent.
Moving motor  1  to  90
Moving motor  2  to  90
Stop response: d064A000004005A0000000000110200001E01
Serial port closed: True


#### Flipper settings

In [137]:

flipper1 = flipper_utils.Controller(
    'COM8',                 # pick the correct COM port
    name='MFF101',          # optional change name
    verbose=True,           # optional verbose
    very_verbose=False      # optional very_verbose
    )

#flipper2 = flipper_utils.Controller(
    #'COM8',                 # pick the correct COM port
    #name='MFF101',          # optional change name
    #verbose=True,           # optional verbose
    #very_verbose=False      # optional very_verbose
    #)

MFF101: opening... done.
MFF101: getting info
MFF101: -> model number  = MFF002  
MFF101: -> type          = 16
MFF101: -> serial number = 37011596
MFF101: -> firmware version = 65539
MFF101: -> hardware version = 2
MFF101: getting position...
MFF101: -> position = 1


In [141]:
flipper_utils.state(flip_ID = flipper1, block=True)

AttributeError: module 'flipper_utils' has no attribute 'state'

#### Remote functions:


In [82]:
#Quick fetch of Displacement from SERF321 computer

def fetch_remote_dis():
    result = subprocess.run([
        "ssh",
        "serf321@10.46.40.133",
        "python --version &&",
        r'python C:\Users\SERF321\Documents\YuGroupCode\Time_Tagger\Swabian\displacement_calib_nogate.py'
    ],
        stdout = subprocess.PIPE,
        stderr = subprocess.PIPE,
        #capture_output=True,
        text=True
    )

    alpha_sum = float(result.stdout.splitlines()[-1])
    #print("Alpha Sum:", alpha_sum)
    print(result.stderr)
    return alpha_sum

#### Dark measurement

Flip down the flipper if it is not already down

In [134]:
flp1_pos = flipper1.get_position()
#flp2_pos = flipper2.get_position()

if flp1_pos == 1:
    print("Flipper 1 is in down position -- blocked!")
else:
    flipper1.flip(1)     ### down position
    print("Flipper 1 is flipped down")

#if flp2_pos == 1:
    #print("Flipper 2 is in down position -- blocked!")
#else:
    #flipper1.flip(1)     ### down position
    #print("Flipper 2 is flipped down")
    
# close
#flipper1.close()
#flipper2.close()

#Take measurements remotely with SSH
d_dk = fetch_remote_dis()
print("Dark displacement from remote measurement:", d_dk)

MFF101: getting position...
MFF101: -> position = 1
Flipper 1 is in down position -- blocked!


KeyboardInterrupt: 

#### Block displacment, set both HWP's
Calibrate matrix eta to HWP angle  

In [94]:
#code to unblock first flipper and block second flipper
flp1_pos = flipper1.get_position()
#flp2_pos = flipper2.get_position()

if flp1_pos == 1:
    print("Flipper 1 is in down position -- blocked!")
else:
    flipper1.flip(0)     ### up position
    print("Flipper 1 is flipped up")

#if flp2_pos == 0:
    #print("Flipper 2 is in up position -- unblocked!")
#else:
#    flipper2.flip(0)     ### up position
#   print("Flipper 2 is flipped up")

# close
#flipper1.close()
#flipper2.close()

MFF101: getting position...
MFF101: -> position = 1
Flipper 1 is in down position -- blocked!


In [96]:
#adjust HWP to zero-angle which is eta = 1
#set_motor_angle(m1ang_init, m2ang_init)

eta_value = 1

def eta_to_angle(eta_value):
    theta =np.arccos(np.sqrt(eta_value))/2
    print("theta (in degrees)= ", np.degrees(theta))
    hwp_ang1 = m1ang_init + np.degrees(theta)   # for matrix w
    hwp_ang2 = m2ang_init - np.degrees(theta)   # for matrix inversed w
    return hwp_ang1, hwp_ang2

hwp_ang1, hwp_ang2 = eta_to_angle(eta_value)
set_motor_angle(hwp_ang1, hwp_ang2)


theta (in degrees)=  0.0
Opening COM6...
Port open: True
Initialization commands sent.
Moving motor  1  to  8.5
Moving motor  2  to  0.98
Stop response: d064A000000F8000000001900250200001D01
Serial port closed: True


#### Check SPDC heralding
measure effective single photon efficiency

#### Block SPDC, inject Displacment
measure photon number distribution to calibrate mu

In [ ]:
#code to unblock first flipper and block second flipper
flp1_pos = flipper1.get_position()
#flp2_pos = flipper2.get_position()

if flp1_pos == 1:
    print("Flipper 1 is in down position -- blocked!")
else:
    flipper1.flip(0)     ### up position
    print("Flipper 1 is flipped up")

#if flp2_pos == 0:
    #print("Flipper 2 is in up position -- unblocked!")
#else:
#    flipper2.flip(0)     ### up position
#   print("Flipper 2 is flipped up")

# close
#flipper1.close()
#flipper2.close()

#Take measurements remotely with SSH
d_set = fetch_remote_dis()
print("Displacement value from remote measurement:", d_set)

MFF101: getting position...
MFF101: -> position = 1
Flipper 1 is in down position -- blocked!
MFF101: closing... done.



0.051449999999999996

#### Open both flippers
measure joint two-port distribution

In [ ]:
#code to unblock first flipper and block second flipper
flp1_pos = flipper1.get_position()
#flp2_pos = flipper2.get_position()

if flp1_pos == 1:
    print("Flipper 1 is in down position -- blocked!")
else:
    flipper1.flip(0)     ### up position
    print("Flipper 1 is flipped up")

#if flp2_pos == 0:
    #print("Flipper 2 is in up position -- unblocked!")
#else:
#    flipper2.flip(0)     ### up position
#   print("Flipper 2 is flipped up")

# close
#flipper1.close()
#flipper2.close()